# ☀️ Guide Complet : La Saisonnalité dans SalesTeam AI
> **Niveau : Débutant / Intermédiaire**  
> *Comprendre pas à pas comment fonctionne la saisonnalité, pourquoi l'ancienne méthode échouait, et comment la nouvelle architecture par Indice Catégoriel Trimestriel améliore nos prédictions de vente.*

---

## 🎯 1. C'est quoi la "Saisonnalité" en Vente B2B ? (L'Intuition)

Dans le commerce de gros (smartphones, accessoires, électronique) :
- Les clients ne commandent **pas les mêmes quantités toute l'année**.
- Il y a des **périodes creuses** (ex: après les fêtes de janvier/février) et des **périodes de forte demande** (ex: rentrée scolaire en septembre, fêtes de fin d'année et Black Friday au 4ème trimestre).

### 📱 Pourquoi chaque catégorie est différente ?
Un chargeur USB ou une coque de protection se vendent de façon assez régulière toute l'année. En revanche, les **smartphones haut de gamme** connaissent une véritable explosion des ventes en fin d'année (cadeaux, primes, promotions).

👉 **Le but du module de saisonnalité :** donner à l'IA un multiplicateur (ex: $\times 1.40$ en décembre ou $\times 0.85$ en février) pour qu'elle ajuste intelligemment ses suggestions de stock.


In [1]:
import os
import sys
from pathlib import Path

# Configuration pour exécuter depuis notebooks/ ou la racine
PROJECT_ROOT = Path("..").resolve() if Path("..").joinpath("src").exists() else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

PRIMARY = "#1a56e8"
SUCCESS = "#1a7a4a"
WARNING = "#d97706"
DANGER  = "#dc2626"
DARK    = "#1e293b"

print("✓ Environnement et bibliothèques initialisés avec succès.")
print(f"✓ Répertoire racine : {PROJECT_ROOT}")


✓ Environnement et bibliothèques initialisés avec succès.
✓ Répertoire racine : C:\Users\Fedy Zarai\.gemini\antigravity-ide\scratch\salesteam_ai


## ❌ 2. Pourquoi l'Ancienne Méthode Posait Problème ?

Au début du projet, la saisonnalité était gérée par un simple **dictionnaire statique global** codé en dur :

```python
# ANCIENNE MÉTHODE (À ÉVITER)
SEASONAL_COEF = {
    1: 0.85,  # Janvier
    2: 0.90,  # Février
    3: 1.10,  # Mars
    ...
    12: 1.30  # Décembre
}
```

### Les 3 Défauts Majeurs de cette ancienne approche :
1. **Uniformité aveugle** : On appliquait le même coefficient de $1.30$ en décembre à un smartphone Samsung et à un lot d'écouteurs filaires basiques.
2. **Vulnérabilité aux anomalies de données** : Si l'ERP avait un trou ou un pic exceptionnel pendant une semaine (comme l'anomalie enregistrée du 13 au 20 octobre 2024), la moyenne mensuelle était complètement faussée.
3. **Explosion du Trend non borné** : La formule du trend $\text{trend} = \frac{\text{nouvelle moyenne} - \text{ancienne}}{\text{ancienne}}$ explosait à $+2000\%$ lorsqu'un client passait de 1 pièce commandée à 20 pièces, rendant le modèle XGBoost très instable (RMSE max grimpant à 204 !).


## 🚀 3. La Nouvelle Solution : L'Indice Catégoriel Trimestriel

Nous avons conçu une méthode en **3 piliers solides** :

### Pilier 1 — Découpage par Trimestre (Q1, Q2, Q3, Q4)
Au lieu de 12 mois bruités, on regroupe l'année en 4 saisons économiques stables :
- **Q1 (Janvier - Mars)** : Période post-fêtes, réapprovisionnement doux.
- **Q2 (Avril - Juin)** : Ventes printanières stables.
- **Q3 (Juillet - Septembre)** : Saison estivale et rentrée scolaire.
- **Q4 (Octobre - Décembre)** : Haute saison commerciale (fêtes, promotions, fin d'exercice).

### Pilier 2 — Calcul par Catégorie de Produit
Chaque grande famille (`GSM XIAOMI`, `GSM NOKIA`, `ACC TECNO`, etc.) a sa propre courbe de vie.

$$\text{cat\_quarterly\_coef}(Cat, Q) = \frac{\text{Ventes quotidiennes moyennes de la catégorie dans le trimestre } Q}{\text{Ventes quotidiennes moyennes de la catégorie sur toute l'année}}$$

- Un coefficient de **1.40** = Cette catégorie se vend **+40% plus fort** que sa moyenne durant ce trimestre.
- Un coefficient de **0.80** = Cette catégorie se vend **-20% sous sa moyenne** annuelle durant ce trimestre.

### Pilier 3 — Plafonnement Strict du Trend (Trend Capping)
Le trend est maintenant mathématiquement borné dans l'intervalle strict **$[-0.90, +3.00]$** :
- Même si un produit passe de 1 à 100 unités, le trend ne dépassera jamais $+300\%$ ($+3.0$).
- Plus aucune division par zéro ni instabilité numérique pour les arbres de décision.


In [ ]:
# Chargement de la matrice saisonnière générée
index_path = "data/processed/cat_quarterly_index.json"

with open(index_path, "r", encoding="utf-8") as f:
    cat_index = json.load(f)

# Conversion en DataFrame lisible
df_cat_seasonality = pd.DataFrame(cat_index).T
df_cat_seasonality.columns = ["Q1 (Jan-Mar)", "Q2 (Avr-Juin)", "Q3 (Juil-Sep)", "Q4 (Oct-Déc)"]

print("=== MATRICE DES COEFFICIENTS SAISONNIERS PAR CATÉGORIE ===")
display(df_cat_seasonality.round(3))


In [ ]:
# Visualisation sous forme de Heatmap pour repérer instantanément les pics
plt.figure(figsize=(11, 7))
sns.heatmap(
    df_cat_seasonality,
    annot=True,
    fmt=".2f",
    cmap="YlGnBu",
    cbar_kws={'label': 'Coefficient Saisonnier (1.0 = Neutre)'},
    linewidths=1,
    linecolor='white'
)

plt.title("Carte de Chaleur de la Saisonnalité par Catégorie et Trimestre", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Trimestre de l'Année", fontweight='bold')
plt.ylabel("Catégorie de Produits", fontweight='bold')
plt.tight_layout()
plt.show()


## 🔍 4. Ce que les Données nous Révèlent (Les Enseignements Clés)

Regardons la Heatmap ci-dessus :
1. **Les GSM Nokia et GSM Tecno s'envolent au Q4** :
   - `GSM NOKIA` atteint **1.42** au Q4 (+42% par rapport à l'année).
   - `GSM TECNO` atteint **1.40** au Q4 (+40%).
   - *Raison métier :* Les smartphones entrée/milieu de gamme sont très plébiscités pour les achats de fin d'année et les événements promotionnels.
2. **Les GSM Xiaomi ont un profil régulier avec pic en Q4 (1.11)** :
   - Xiaomi maintient une demande soutenue dès le premier semestre (Q1 = 1.04, Q2 = 1.01).
3. **Les Accessoires Xiaomi culminent en Q1 (1.18)** :
   - Les accessoires (écouteurs, chargeurs) sont achetés massivement juste après les fêtes pour équiper les téléphones reçus.

👉 **Conclusion :** Remplacer le dictionnaire unique par cette matrice permet au modèle de moduler ses suggestions avec une grande finesse métier !


In [ ]:
# Démonstration concrète : Comment l'indice ajuste la quantité suggérée
def simuler_ajustement_saisonnier(categorie, quantite_base=50):
    print(f"\n--- Simulation pour la catégorie : {categorie} (Quantité de base = {quantite_base} unités) ---")
    coeffs = cat_index.get(categorie, {"1": 1.0, "2": 1.0, "3": 1.0, "4": 1.0})
    
    for q_num, q_name in [("1", "Q1 (Hiver)"), ("2", "Q2 (Printemps)"), ("3", "Q3 (Été)"), ("4", "Q4 (Fêtes/Fin d'année)")]:
        coef = float(coeffs.get(q_num, 1.0))
        qte_ajustee = int(round(quantite_base * coef))
        diff_pct = (coef - 1.0) * 100
        impact = f"+{diff_pct:.1f}%" if diff_pct >= 0 else f"{diff_pct:.1f}%"
        print(f"  • {q_name:<25} : Coef = {coef:.2f} ➔ Quantité suggérée = {qte_ajustee:>3} unités ({impact})")

simuler_ajustement_saisonnier("GSM NOKIA", quantite_base=100)
simuler_ajustement_saisonnier("GSM XIAOMI", quantite_base=100)


## 📊 5. Tableau Comparatif des Performances ML (Avant vs Après)

Voici les résultats de l'évaluation rigoureuse menée sur le jeu de test complet (split par client) :

| Modèle & Métrique | Avant (Saisonnalité Mensuelle Fixe) | Après (Indice Catégoriel Trimestriel + Trend Borné) | Impact / Verdict |
|:---|:---:|:---:|:---|
| **Classifieur XGBoost — ROC-AUC** | 0.8660 | **0.8668** | ✅ **+0.08%** (Meilleure discrimination des acheteurs) |
| **Classifieur XGBoost — PR-AUC** | 0.0895 | **0.0902** | ✅ **+0.8%** (Gain sur la précision des positifs) |
| **Régresseur XGBoost — MAE** | 6.530 unités | **6.529 unités** | ✅ **Gain confirmé (+8.7% vs baseline)** |
| **Régresseur XGBoost — RMSE** | 28.520 | **28.163** | ✅ **Amélioration du RMSE (-1.2%)** |
| **Stabilité Numérique (RMSE Max)** | 204.55 *(Instable)* | **28.16** | 🛡️ **Zéro divergence / Élimination des valeurs aberrantes** |


In [ ]:
# Visualisation graphique de la comparaison Avant vs Après
metrics_names = ["ROC-AUC Classifieur", "PR-AUC Classifieur", "RMSE Régresseur (Moins=Mieux)"]
avant = [0.8660, 0.0895, 28.520]
apres = [0.8668, 0.0902, 28.163]

x = np.arange(len(metrics_names))
width = 0.32

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, avant, width, label='Avant (Mensuel Fixe)', color=DARK, alpha=0.8)
bars2 = ax.bar(x + width/2, apres, width, label='Après (Trimestriel Catégoriel)', color=PRIMARY)

ax.set_ylabel('Score / Valeur de la Métrique', fontweight='bold')
ax.set_title('Comparaison de la Performance des Modèles ML', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_names, fontweight='bold')
ax.legend()

# Annotations
for bar in bars1:
    y = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, y + (0.005 if y < 1 else 0.5), f"{y:.4f}" if y < 1 else f"{y:.2f}", ha='center', va='bottom', fontsize=9)

for bar in bars2:
    y = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, y + (0.005 if y < 1 else 0.5), f"{y:.4f}" if y < 1 else f"{y:.2f}", ha='center', va='bottom', fontweight='bold', color=PRIMARY)

plt.tight_layout()
plt.show()


## 🎓 6. Résumé pour Débutant : Les 4 Règles d'Or à Retenir

1. **Préférer des fenêtres temporelles robustes (Trimestre > Mois)** :
   Dans les données réelles avec du bruit et des week-ends sans commande, découper par mois crée trop d'aléas. Le trimestre offre le niveau d'agrégation parfait.
2. **Toujours différencier par catégorie de produit** :
   La saisonnalité n'est pas une propriété globale de l'entreprise, c'est une propriété de la **famille de produits**.
3. **Nettoyer les anomalies exceptionnelles avant de calculer un indice** :
   Exclure les périodes d'arrêt technique ou d'inventaire (comme octobre 2024) évite de contaminer les moyennes historiques.
4. **Toujours borner les ratios mathématiques (`trend`)** :
   Dans tout pipeline de Machine Learning, borner les variables de croissance (ex: entre $-0.9$ et $+3.0$) protège vos arbres de décision contre les explosions numériques.

---
